In [2]:
"""
PROJECT 1: SUPERSTORE SALES DASHBOARD
Complete automated analysis from data cleaning to SQL queries
Author: Data Analytics Internship Project
"""

import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime
import os

In [3]:
print("=" * 80)
print("🚀 SUPERSTORE SALES ANALYSIS - AUTOMATED PIPELINE")
print("=" * 80)

🚀 SUPERSTORE SALES ANALYSIS - AUTOMATED PIPELINE


In [4]:
print("\n📂 PHASE 1: LOADING DATA...")

# Load the CSV file
try:
    df = pd.read_csv("/Users/afxwqk/Downloads/Sample - Superstore.csv", encoding='latin-1')
    print(f"✅ Data loaded successfully!")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
except FileNotFoundError:
    print("❌ Error: 'Sample - Superstore.csv' not found!")
    print("   Download from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final")
    exit()

# Display initial info
print("\n📋 Dataset Overview:")
print(df.head())
print("\n📊 Data Types:")
print(df.dtypes)
print("\n❓ Missing Values:")
print(df.isnull().sum())


📂 PHASE 1: LOADING DATA...
✅ Data loaded successfully!
   Shape: 9994 rows × 21 columns

📋 Dataset Overview:
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort La

In [5]:
print("\n🧹 PHASE 2: CLEANING DATA...")

# Convert date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

# Extract year, month, quarter
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%B')
df['Quarter'] = df['Order Date'].dt.quarter
df['Week'] = df['Order Date'].dt.isocalendar().week

# Calculate days to ship
df['Days_to_Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

# Calculate profit margin
df['Profit_Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Remove duplicates
initial_rows = len(df)
df.drop_duplicates(inplace=True)
duplicates_removed = initial_rows - len(df)

print(f"✅ Date columns converted")
print(f"✅ New columns created (Year, Month, Profit_Margin, etc.)")
print(f"✅ Duplicates removed: {duplicates_removed}")
print(f"✅ Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# Save cleaned data
df.to_csv("superstore_clean.csv", index=False)
print(f"\n Cleaned data saved as: superstore_clean.csv")


🧹 PHASE 2: CLEANING DATA...
✅ Date columns converted
✅ New columns created (Year, Month, Profit_Margin, etc.)
✅ Duplicates removed: 0
✅ Final dataset: 9994 rows × 28 columns

💾 Cleaned data saved as: superstore_clean.csv


In [6]:
print("\n" + "=" * 80)
print("🗄️  PHASE 3: SQL ANALYSIS")
print("=" * 80)

# Create SQLite database
conn = sqlite3.connect("superstore.db")
df.to_sql("sales", conn, if_exists="replace", index=False)
print("\n✅ SQLite database created: superstore.db")


# QUERY 1: REVENUE & PROFIT BY REGION

print("\n" + "-" * 80)
print("QUERY 1: REVENUE & PROFIT BY REGION")
print("-" * 80)

query1 = """
SELECT Region,
       ROUND(SUM(Sales), 2) AS Total_Revenue,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct,
       COUNT(*) AS Number_of_Orders
FROM sales
GROUP BY Region
ORDER BY Total_Revenue DESC;
"""

result1 = pd.read_sql_query(query1, conn)
print(result1.to_string(index=False))
result1.to_csv("01_revenue_by_region.csv", index=False)
print("\n Saved as: 01_revenue_by_region.csv")


🗄️  PHASE 3: SQL ANALYSIS

✅ SQLite database created: superstore.db

--------------------------------------------------------------------------------
QUERY 1: REVENUE & PROFIT BY REGION
--------------------------------------------------------------------------------
 Region  Total_Revenue  Total_Profit  Profit_Margin_Pct  Number_of_Orders
   West      725457.82     108418.45              14.94              3203
   East      678781.24      91522.78              13.48              2848
Central      501239.89      39706.36               7.92              2323
  South      391721.91      46749.43              11.93              1620

💾 Saved as: 01_revenue_by_region.csv


In [7]:
print("\n" + "-" * 80)
print("QUERY 2: TOP 10 PRODUCT SUB-CATEGORIES BY SALES")
print("-" * 80)

query2 = """
SELECT "Sub-Category",
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct,
       COUNT(*) AS Order_Count
FROM sales
GROUP BY "Sub-Category"
ORDER BY Total_Sales DESC
LIMIT 10;
"""

result2 = pd.read_sql_query(query2, conn)
print(result2.to_string(index=False))
result2.to_csv("02_top_products_by_sales.csv", index=False)
print("\n Saved as: 02_top_products_by_sales.csv")


--------------------------------------------------------------------------------
QUERY 2: TOP 10 PRODUCT SUB-CATEGORIES BY SALES
--------------------------------------------------------------------------------
Sub-Category  Total_Sales  Total_Profit  Profit_Margin_Pct  Order_Count
      Phones    330007.05      44515.73              13.49          889
      Chairs    328449.10      26590.17               8.10          617
     Storage    223843.61      21278.83               9.51          846
      Tables    206965.53     -17725.48              -8.56          319
     Binders    203412.73      30221.76              14.86         1523
    Machines    189238.63       3384.76               1.79          115
 Accessories    167380.32      41936.64              25.05          775
     Copiers    149528.03      55617.82              37.20           68
   Bookcases    114880.00      -3472.56              -3.02          228
  Appliances    107532.16      18138.01              16.87          4

In [8]:
print("\n" + "-" * 80)
print("QUERY 3: YEAR-OVER-YEAR REVENUE GROWTH")
print("-" * 80)

query3 = """
SELECT Year,
       ROUND(SUM(Sales), 2) AS Total_Revenue,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       COUNT(*) AS Order_Count,
       ROUND(AVG(Sales), 2) AS Avg_Order_Value
FROM sales
GROUP BY Year
ORDER BY Year;
"""

result3 = pd.read_sql_query(query3, conn)
print(result3.to_string(index=False))
result3.to_csv("03_yearly_revenue.csv", index=False)

# Calculate YoY growth
print("\n📈 Year-over-Year Growth:")
for i in range(1, len(result3)):
    prev_year_revenue = result3.iloc[i-1]['Total_Revenue']
    curr_year_revenue = result3.iloc[i]['Total_Revenue']
    growth = ((curr_year_revenue - prev_year_revenue) / prev_year_revenue * 100)
    print(f"   {int(result3.iloc[i]['Year'])}: {growth:+.2f}%")

print("\n Saved as: 03_yearly_revenue.csv")



--------------------------------------------------------------------------------
QUERY 3: YEAR-OVER-YEAR REVENUE GROWTH
--------------------------------------------------------------------------------
 Year  Total_Revenue  Total_Profit  Order_Count  Avg_Order_Value
 2014      484247.50      49543.97         1993           242.97
 2015      470532.51      61618.60         2102           223.85
 2016      609205.60      81795.17         2587           235.49
 2017      733215.26      93439.27         3312           221.38

📈 Year-over-Year Growth:
   2015: -2.83%
   2016: +29.47%
   2017: +20.36%

💾 Saved as: 03_yearly_revenue.csv


In [12]:
print("\n" + "-" * 80)
print("QUERY 4: PROFIT BY CUSTOMER SEGMENT")
print("-" * 80)

query4 = """
SELECT Segment,
       COUNT(DISTINCT "Customer ID") AS Num_Customers,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct,
       ROUND(AVG(Profit), 2) AS Avg_Profit_Per_Order
FROM sales
GROUP BY Segment
ORDER BY Total_Profit DESC;
"""

result4 = pd.read_sql_query(query4, conn)
print(result4.to_string(index=False))
result4.to_csv("04_profit_by_segment.csv", index=False)
print("\n Saved as: 04_profit_by_segment.csv")



--------------------------------------------------------------------------------
QUERY 4: PROFIT BY CUSTOMER SEGMENT
--------------------------------------------------------------------------------
    Segment  Num_Customers  Total_Sales  Total_Profit  Profit_Margin_Pct  Avg_Profit_Per_Order
   Consumer            409   1161401.34     134119.21              11.55                 25.84
  Corporate            236    706146.37      91979.13              13.03                 30.46
Home Office            148    429653.15      60298.68              14.03                 33.82

 Saved as: 04_profit_by_segment.csv


In [13]:
print("\n" + "-" * 80)
print("QUERY 5: LOSS-MAKING PRODUCTS (RED FLAGS)")
print("-" * 80)
 
query5 = """
SELECT "Product Name",
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct,
       COUNT(*) AS Order_Count
FROM sales
GROUP BY "Product Name"
HAVING SUM(Profit) < 0
ORDER BY SUM(Profit) ASC;
"""
 
result5 = pd.read_sql_query(query5, conn)
if len(result5) > 0:
    print(result5.to_string(index=False))
    result5.to_csv("05_unprofitable_products.csv", index=False)
    print(f"\n⚠️  Found {len(result5)} unprofitable products!")
else:
    print("✅ No loss-making products found!")
 
print("\n Saved as: 05_unprofitable_products.csv")


--------------------------------------------------------------------------------
QUERY 5: LOSS-MAKING PRODUCTS (RED FLAGS)
--------------------------------------------------------------------------------
                                                                        Product Name  Total_Sales  Total_Profit  Profit_Margin_Pct  Order_Count
                                           Cubify CubeX 3D Printer Double Head Print     11099.96      -8879.97             -80.00            3
                                           Lexmark MX611dhe Monochrome Laser Printer     16829.90      -4589.97             -27.27            4
                                           Cubify CubeX 3D Printer Triple Head Print      7999.98      -3839.99             -48.00            1
                            Chromcraft Bull-Nose Wood Oval Conference Tables & Bases      9917.64      -2876.12             -29.00            5
                                Bush Advantage Collection Racetrack Confere

In [14]:
print("\n" + "-" * 80)
print("QUERY 6: SALES BY CATEGORY AND REGION")
print("-" * 80)
 
query6 = """
SELECT Region,
       Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       COUNT(*) AS Order_Count
FROM sales
GROUP BY Region, Category
ORDER BY Region, Total_Sales DESC;
"""
 
result6 = pd.read_sql_query(query6, conn)
print(result6.to_string(index=False))
result6.to_csv("06_sales_by_category_region.csv", index=False)
print("\n Saved as: 06_sales_by_category_region.csv")


--------------------------------------------------------------------------------
QUERY 6: SALES BY CATEGORY AND REGION
--------------------------------------------------------------------------------
 Region        Category  Total_Sales  Total_Profit  Order_Count
Central      Technology    170416.31      33697.43          420
Central Office Supplies    167026.42       8879.98         1422
Central       Furniture    163797.16      -2871.05          481
   East      Technology    264973.98      47462.04          535
   East       Furniture    208291.20       3046.17          601
   East Office Supplies    205516.05      41014.58         1712
  South      Technology    148771.91      19991.83          293
  South Office Supplies    125651.31      19986.39          995
  South       Furniture    117298.68       6771.21          332
   West       Furniture    252612.74      11504.95          707
   West      Technology    251991.83      44303.65          599
   West Office Supplies    2208

In [15]:
print("\n" + "-" * 80)
print("QUERY 7: MONTHLY SALES TREND (2017 DATA)")
print("-" * 80)
 
query7 = """
SELECT Year,
       Month,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       COUNT(*) AS Order_Count
FROM sales
WHERE Year = 2017
GROUP BY Year, Month
ORDER BY Month;
"""
 
result7 = pd.read_sql_query(query7, conn)
print(result7.to_string(index=False))
result7.to_csv("07_monthly_sales_2017.csv", index=False)
print("\n Saved as: 07_monthly_sales_2017.csv")
 


--------------------------------------------------------------------------------
QUERY 7: MONTHLY SALES TREND (2017 DATA)
--------------------------------------------------------------------------------
 Year  Month  Total_Sales  Total_Profit  Order_Count
 2017      1     43971.37       7140.44          155
 2017      2     20301.13       1613.87          107
 2017      3     58872.35      14751.89          238
 2017      4     36521.54        933.29          203
 2017      5     44261.11       6342.58          242
 2017      6     52981.73       8223.34          245
 2017      7     45264.42       6952.62          226
 2017      8     63120.89       9040.96          218
 2017      9     87866.65      10991.56          459
 2017     10     77776.92       9275.28          298
 2017     11    118447.82       9690.10          459
 2017     12     83829.32       8483.35          462

 Saved as: 07_monthly_sales_2017.csv


In [16]:
print("\n" + "-" * 80)
print("QUERY 8: TOP 10 CUSTOMERS BY PROFIT")
print("-" * 80)
 
query8 = """
SELECT "Customer Name",
       Segment,
       Region,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       COUNT(*) AS Order_Count,
       ROUND(AVG(Sales), 2) AS Avg_Order_Value
FROM sales
GROUP BY "Customer Name"
ORDER BY Total_Profit DESC
LIMIT 10;
"""
 
result8 = pd.read_sql_query(query8, conn)
print(result8.to_string(index=False))
result8.to_csv("08_top_customers.csv", index=False)
print("\n Saved as: 08_top_customers.csv")
 


--------------------------------------------------------------------------------
QUERY 8: TOP 10 CUSTOMERS BY PROFIT
--------------------------------------------------------------------------------
       Customer Name     Segment  Region  Total_Sales  Total_Profit  Order_Count  Avg_Order_Value
        Tamara Chand   Corporate    West     19052.22       8981.32           12          1587.68
        Raymond Buch    Consumer    East     15117.34       6976.10           18           839.85
        Sanjit Chand    Consumer    West     14142.33       5757.41           22           642.83
        Hunter Lopez    Consumer Central     12873.30       5622.43           11          1170.30
       Adrian Barton    Consumer    West     14473.57       5444.81           20           723.68
        Tom Ashbrook Home Office    East     14595.62       4703.79           10          1459.56
Christopher Martinez    Consumer   South      8954.02       3899.89           10           895.40
       Keith Dawk

In [17]:
print("\n" + "=" * 80)
print("📊 SUMMARY STATISTICS")
print("=" * 80)
 
print(f"""
Total Revenue:        ${df['Sales'].sum():,.2f}
Total Profit:         ${df['Profit'].sum():,.2f}
Overall Profit Margin: {(df['Profit'].sum()/df['Sales'].sum()*100):.2f}%
 
Total Orders:         {len(df):,}
Total Customers:      {df['Customer ID'].nunique():,}
Total Products:       {df['Product Name'].nunique():,}
 
Average Order Value:  ${df['Sales'].mean():,.2f}
Average Profit/Order: ${df['Profit'].mean():,.2f}
 
Years Covered:        {df['Year'].min():.0f} - {df['Year'].max():.0f}
Regions:              {', '.join(df['Region'].unique())}
Categories:           {', '.join(df['Category'].unique())}
""")
 


📊 SUMMARY STATISTICS

Total Revenue:        $2,297,200.86
Total Profit:         $286,397.02
Overall Profit Margin: 12.47%

Total Orders:         9,994
Total Customers:      793
Total Products:       1,850

Average Order Value:  $229.86
Average Profit/Order: $28.66

Years Covered:        2014 - 2017
Regions:              South, West, Central, East
Categories:           Furniture, Office Supplies, Technology



In [18]:
print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE!")
print("=" * 80)
 
print("""
📁 Generated Files:
   1. superstore_clean.csv          (Cleaned dataset)
   2. superstore.db                 (SQLite database)
   3. 01_revenue_by_region.csv       (Revenue analysis by region)
   4. 02_top_products_by_sales.csv   (Top 10 products)
   5. 03_yearly_revenue.csv          (YoY growth trends)
   6. 04_profit_by_segment.csv       (Profit by customer segment)
   7. 05_unprofitable_products.csv   (Loss-making products)
   8. 06_sales_by_category_region.csv (Regional breakdown)
   9. 07_monthly_sales_2017.csv      (Monthly trends)
   10. 08_top_customers.csv          (Top customers)
 
📚 Key Insights to Highlight:
   - Most profitable region: {result1.iloc[0]['Region']} (${result1.iloc[0]['Total_Profit']:,.2f})
   - Top product: {result2.iloc[0]['Sub-Category']} (${result2.iloc[0]['Total_Sales']:,.2f})
   - Revenue trend: Check 03_yearly_revenue.csv
   - Best segment: {result4.iloc[0]['Segment']} (${result4.iloc[0]['Total_Profit']:,.2f})
""")
 
conn.close()
print("✨ Database connection closed. Ready for Tableau!")
print("=" * 80)


✅ ANALYSIS COMPLETE!

📁 Generated Files:
   1. superstore_clean.csv          (Cleaned dataset)
   2. superstore.db                 (SQLite database)
   3. 01_revenue_by_region.csv       (Revenue analysis by region)
   4. 02_top_products_by_sales.csv   (Top 10 products)
   5. 03_yearly_revenue.csv          (YoY growth trends)
   6. 04_profit_by_segment.csv       (Profit by customer segment)
   7. 05_unprofitable_products.csv   (Loss-making products)
   8. 06_sales_by_category_region.csv (Regional breakdown)
   9. 07_monthly_sales_2017.csv      (Monthly trends)
   10. 08_top_customers.csv          (Top customers)

🎯 Next Steps:
   1. Download all CSV files
   2. Connect superstore_clean.csv to Tableau Public
   3. Create 4 visualizations:
      - Bar Chart: Revenue by Region
      - Line Chart: Monthly Sales Trend
      - Treemap: Sales by Sub-Category
      - Scatter: Sales vs Profit
   4. Create interactive dashboard
   5. Publish to Tableau Public
   6. Share link on GitHub & LinkedI